# BoardBridge BD — reproducible Gemma 4 vision call

Multimodal Track, Build With Gemma @Bangladesh. Live app: https://boardbridge-bd.vercel.app · Repo: https://github.com/ai-naymul/boardbridge-bd

This notebook reproduces **stage 1** of the pipeline: a whiteboard photo goes to **`gemma-4-31b-it`** and comes back as a schema-validated `BoardArtifact`. The prompt below is byte-identical to `lib/prompts.ts` in the app.

In [ ]:
!pip install -q google-genai

## API key

The key is read from an environment variable / Kaggle Secret. **Never paste a key into a notebook cell** — its output is public.

In [ ]:
import os, json, base64, time
from google import genai
from google.genai import types

# Kaggle: Add-ons -> Secrets -> GEMINI_API_KEY. Colab: userdata. Local: env var.
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ["GEMINI_API_KEY"]

MODEL = "gemma-4-31b-it"
client = genai.Client(api_key=API_KEY)
print("model:", MODEL)

## The extraction system prompt (verbatim from `lib/prompts.ts`)

In [ ]:
EXTRACTION_SYSTEM = """You transcribe photographed classroom whiteboards from Bangladesh into structured JSON.

FAITHFULNESS RULES — these override everything else:
- Transcribe EXACTLY what is written on the board. Preserve Bangla, English, Bangla-English
  code-switching, Bangla numerals, symbols and mathematical notation as they appear.
- Do NOT translate. Do NOT correct spelling, grammar or notation.
- Keep English technical vocabulary in English.

UNCERTAINTY RULES:
- If a span is smudged or illegible, transcribe your best guess, list that span in
  "uncertainSpans", and set that region confidence to "low". Never silently guess.

EMPTY / UNUSABLE INPUT:
- If the image contains no legible board content, return "imageQuality":"unusable", an
  EMPTY "regions" array, and explain why in "warnings". NEVER invent board content.

STRUCTURE:
- One region per visual block. "order" follows reading order. Ids are r1, r2, r3, ...

Output JSON only."""
print(len(EXTRACTION_SYSTEM), "chars")

## Schema sent as `responseJsonSchema`

In [ ]:
SCHEMA = {
  "type": "object",
  "properties": {
    "title": {"type": "string"},
    "detectedLanguages": {"type": "array", "items": {"type": "string"}},
    "imageQuality": {"type": "string", "enum": ["good", "fair", "poor", "unusable"]},
    "regions": {"type": "array", "items": {"type": "object", "properties": {
        "id": {"type": "string"}, "order": {"type": "integer"},
        "type": {"type": "string"}, "transcription": {"type": "string"},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "uncertainSpans": {"type": "array", "items": {"type": "string"}}},
      "required": ["id", "order", "type", "transcription", "confidence"]}},
    "warnings": {"type": "array", "items": {"type": "string"}}},
  "required": ["title", "detectedLanguages", "imageQuality", "regions"]}

## The board

Boards are author-created for this submission (`eval/make_boards.py` in the repo) — not photos of real classrooms, not scraped. Swap in your own photo below.

In [ ]:
IMAGE_PATH = "../eval/images/mixed_notes.jpg"   # or any whiteboard photo
image_bytes = open(IMAGE_PATH, "rb").read()
print(len(image_bytes), "bytes")

## The call

In [ ]:
t0 = time.time()
resp = client.models.generate_content(
    model=MODEL,
    contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
              "Transcribe this whiteboard photograph into the BoardArtifact JSON structure."],
    config=types.GenerateContentConfig(
        system_instruction=EXTRACTION_SYSTEM,
        temperature=0,
        response_mime_type="application/json",
        response_json_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal")))
print(f"latency: {time.time()-t0:.1f}s")

## Structural decode + validation

Gemma occasionally appends trailing prose or wraps the object in a single-element array, so we decode the **first complete JSON value** rather than calling `json.loads` on the whole string. This mirrors `firstJsonValue` in `lib/gemma.ts`. It is a structural decode, not a regex scrape.

In [ ]:
def first_json_value(text):
    s = text.strip()
    if s.startswith("```"):
        s = s.split("
", 1)[1].rsplit("```", 1)[0].strip()
    start = min(i for i in (s.find("{"), s.find("[")) if i != -1)
    obj, _ = json.JSONDecoder().raw_decode(s[start:])
    if isinstance(obj, list):
        obj = next(x for x in obj if isinstance(x, dict))
    return obj

artifact = first_json_value(resp.text)

REQUIRED = {"title", "detectedLanguages", "imageQuality", "regions"}
assert REQUIRED <= set(artifact), f"missing keys: {REQUIRED - set(artifact)}"
for r in artifact["regions"]:
    assert {"id", "order", "type", "transcription", "confidence"} <= set(r)
    assert r["confidence"] in ("high", "medium", "low")
print("schema OK —", len(artifact["regions"]), "regions")

## Sample output

In [ ]:
print("title:    ", artifact["title"])
print("languages:", artifact["detectedLanguages"])
print("quality:  ", artifact["imageQuality"])
print()
for r in artifact["regions"]:
    flag = "  <-- UNCERTAIN" if r.get("uncertainSpans") else ""
    print(f"[{r['id']}] {r['type']:<16} conf={r['confidence']:<6}{flag}")
    print("     ", r["transcription"][:110].replace("
", " / "))

## The negative control — the case that matters most

An unreadable board must return `imageQuality: "unusable"` with **zero** regions, not invented lecture notes. A student who missed the class cannot tell a correct summary from a confident invention.

In [ ]:
neg = open("../eval/images/blurred_negative.jpg", "rb").read()
r2 = client.models.generate_content(
    model=MODEL,
    contents=[types.Part.from_bytes(data=neg, mime_type="image/jpeg"),
              "Transcribe this whiteboard photograph into the BoardArtifact JSON structure."],
    config=types.GenerateContentConfig(
        system_instruction=EXTRACTION_SYSTEM, temperature=0,
        response_mime_type="application/json", response_json_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal")))
neg_art = first_json_value(r2.text)
print("imageQuality:", neg_art["imageQuality"])
print("regions:     ", len(neg_art["regions"]))
print("warnings:    ", neg_art.get("warnings"))
assert neg_art["imageQuality"] == "unusable" and not neg_art["regions"], "FABRICATION DETECTED"
print("
PASS — refused to fabricate content from an unreadable board.")

## Results and limitations

Measured on n = 3 boards (full method + raw outputs in `eval/` in the repo):

| Board | Quality | Regions | Key-fact recall | Source-support | Extract |
|---|---|---|---|---|---|
| Bangla+English hash tables | good | 10 | 10/10 | 100 % (13/13) | 28.3 s |
| BFS flowchart + pseudocode | good | 12 | 8/8 | 100 % (9/9) | 32.5 s |
| Unreadable control | unusable | 0 | — | — | 3.2 s |

**Limitations.** n = 3 supports no statistical claim. The boards are author-generated renders with photo-style degradation, not real marker handwriting, so these are an upper bound. No formal user testing was conducted. One real transcription error (হ্যাঁ → হাঁ) went unflagged — documented in `eval/results.md`. Latency of ~30 s per stage is slow for mobile data.

**Stage 2** (verified transcript → study pack) is in `app/api/studypack/route.ts`; run the whole pipeline against any instance with `python3 scripts/e2e.py <base-url>`.